# Goal: understand the dataset structure and issues that affect prompting + evaluation.


In [ ]:
import os
import json
import numpy as np
import pandas as pd

from datasets import load_dataset

# ---------- Config ----------
SEED = 42
OUTPUT_DIR = "data_exploration"
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.random.seed(SEED)


# ---------- 1) Load dataset ----------
# Dolly-15k on Hugging Face
ds = load_dataset("databricks/databricks-dolly-15k")
df = ds["train"].to_pandas()

print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head(3)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Rows: 15011
Columns: ['instruction', 'context', 'response', 'category']


,instruction,context,response,category
0,When did Virgin Australia start operating?,"Virgin Australia, the trading name of Virgin A...",Virgin Australia commenced services on 31 Augu...,closed_qa
1,Which is a species of fish? Tope or Rope,,Tope,classification
2,Why can camels survive for long without water?,,Camels use the fat in their humps to keep them...,open_qa


In [ ]:
# ---------- 2) Basic checks ----------
# Make sure required fields exist
required_cols = {"instruction", "response", "context", "category"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Quick null checks
null_stats = df[list(required_cols)].isna().mean().sort_values(ascending=False)
print("Null rate per column:\n", null_stats)

# Basic preview of categories
print("Unique categories:", df["category"].nunique())
df["category"].value_counts().head(10)

Null rate per column:
 context        0.0
response       0.0
category       0.0
instruction    0.0
dtype: float64
Unique categories: 8


,count
category,
open_qa,3742
general_qa,2191
classification,2136
closed_qa,1773
brainstorming,1766
information_extraction,1506
summarization,1188
creative_writing,709


In [ ]:
# ---------- 3) Context availability ----------
df["context_clean"] = df["context"].fillna("").astype(str).str.strip()
df["has_context"] = df["context_clean"].ne("")

context_rate = df["has_context"].mean()
print(f"Has context rate: {context_rate:.3f}")
df["has_context"].value_counts()

Has context rate: 0.298


,count
has_context,
False,10544
True,4467


In [ ]:
# ---------- 4) Length analysis ----------
def char_len(s: str) -> int:
    return len(s)

def word_len(s: str) -> int:
    s = (s or "").strip()
    if not s:
        return 0
    return len(s.split())

df["instruction_char_len"] = df["instruction"].fillna("").astype(str).apply(char_len)
df["response_char_len"] = df["response"].fillna("").astype(str).apply(char_len)
df["context_char_len"] = df["context_clean"].apply(char_len)

df["instruction_word_len"] = df["instruction"].fillna("").astype(str).apply(word_len)
df["response_word_len"] = df["response"].fillna("").astype(str).apply(word_len)
df["context_word_len"] = df["context_clean"].apply(word_len)

length_summary = df[[
    "instruction_word_len", "response_word_len", "context_word_len",
    "instruction_char_len", "response_char_len", "context_char_len"
]].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).T

length_summary

,count,mean,std,min,50%,75%,90%,95%,99%,max
instruction_word_len,15011.0,12.378656,17.048248,1.0,10.0,14.0,21.0,27.0,47.0,841.0
response_word_len,15011.0,60.115515,98.312539,1.0,31.0,73.0,137.0,202.0,433.9,4274.0
context_word_len,15011.0,56.667510,153.330154,0.0,0.0,66.0,176.0,277.0,634.0,3796.0
instruction_char_len,15011.0,71.839384,134.679885,4.0,54.0,81.0,125.0,162.0,277.0,11698.0
response_char_len,15011.0,358.104190,591.581031,1.0,186.0,430.0,803.0,1194.5,2648.8,26018.0
context_char_len,15011.0,352.898008,956.287842,0.0,0.0,408.0,1082.0,1725.5,4022.8,23505.0


In [ ]:
# ---------- 5) Category distribution + imbalance ----------
cat_counts = df["category"].value_counts()
cat_dist = (cat_counts / len(df)).rename("ratio")

cat_table = pd.concat([cat_counts.rename("count"), cat_dist], axis=1)
cat_table.head(15)

,count,ratio
category,,
open_qa,3742,0.249284
general_qa,2191,0.145960
classification,2136,0.142296
closed_qa,1773,0.118113
brainstorming,1766,0.117647
information_extraction,1506,0.100326
summarization,1188,0.079142
creative_writing,709,0.047232


In [ ]:
# ---------- 6) Find "hard" cases (heuristics) ----------
# Hard cases can be:
# - long context
# - long instruction
# - very long expected response
ctx_p95 = df["context_word_len"].quantile(0.95)
ins_p95 = df["instruction_word_len"].quantile(0.95)
rsp_p95 = df["response_word_len"].quantile(0.95)

df["flag_long_context"] = df["context_word_len"] >= ctx_p95
df["flag_long_instruction"] = df["instruction_word_len"] >= ins_p95
df["flag_long_response"] = df["response_word_len"] >= rsp_p95
df["flag_no_context"] = ~df["has_context"]

df["difficulty_score"] = (
    df["flag_long_context"].astype(int) +
    df["flag_long_instruction"].astype(int) +
    df["flag_long_response"].astype(int) +
    df["flag_no_context"].astype(int)
)

df["difficulty_score"].value_counts().sort_index()

,count
difficulty_score,
0,3490
1,10302
2,1156
3,63


In [ ]:
# ---------- 7) Create stratified samples for later experiments ----------

def stratified_sample(data: pd.DataFrame, group_col: str, n_total: int, seed: int = 42) -> pd.DataFrame:
    groups = data[group_col].value_counts()
    ratios = groups / len(data)

    alloc = (ratios * n_total).round().astype(int)
    alloc[alloc < 1] = 1

    diff = int(alloc.sum() - n_total)
    if diff != 0:
        order = alloc.sort_values(ascending=False).index.tolist()
        i = 0
        while diff != 0 and i < len(order) * 10:
            c = order[i % len(order)]
            if diff > 0 and alloc[c] > 1:
                alloc[c] -= 1
                diff -= 1
            elif diff < 0:
                alloc[c] += 1
                diff += 1
            i += 1

    parts = []
    for cat, k in alloc.items():
        sub = data[data[group_col] == cat]
        k = min(k, len(sub))
        parts.append(sub.sample(n=k, random_state=seed))

    out = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    return out

PROMPT_SAMPLE_N = 200
EVAL_SAMPLE_N = 40

prompt_df = stratified_sample(df, "category", PROMPT_SAMPLE_N, seed=SEED)
eval_df = stratified_sample(df, "category", EVAL_SAMPLE_N, seed=SEED)

print("Prompt sample:", len(prompt_df), "Eval sample:", len(eval_df))
prompt_df["category"].value_counts().head(10), eval_df["category"].value_counts().head(10)

Prompt sample: 200 Eval sample: 40


(category
 open_qa                   50
 general_qa                29
 classification            28
 closed_qa                 24
 brainstorming             24
 information_extraction    20
 summarization             16
 creative_writing           9
 Name: count, dtype: int64,
 category
 open_qa                   9
 classification            6
 general_qa                6
 brainstorming             5
 closed_qa                 5
 information_extraction    4
 summarization             3
 creative_writing          2
 Name: count, dtype: int64)

In [ ]:
# Save EDA tables
cat_table.to_csv(os.path.join(OUTPUT_DIR, "category_distribution.csv"), index=True)
length_summary.to_csv(os.path.join(OUTPUT_DIR, "length_summary.csv"), index=True)

prompt_df.to_csv(os.path.join(OUTPUT_DIR, "prompt_sample.csv"), index=False)
eval_df.to_csv(os.path.join(OUTPUT_DIR, "eval_sample.csv"), index=False)

hard_df = df.sort_values("difficulty_score", ascending=False).head(50)
hard_df.to_csv(os.path.join(OUTPUT_DIR, "hard_examples_top50.csv"), index=False)

print("Saved files to:", OUTPUT_DIR)

Saved files to: data_exploration


In [ ]:

notes = []

notes.append(f"Dataset size: {len(df)} rows, {df['category'].nunique()} categories.")
notes.append(f"Context availability: {df['has_context'].mean():.1%} of samples contain non-empty context.")
notes.append(f"Category imbalance: top category ratio = {cat_dist.iloc[0]:.1%} (category: {cat_dist.index[0]}).")

notes.append(f"Instruction length (words) median={df['instruction_word_len'].median():.0f}, p95={df['instruction_word_len'].quantile(0.95):.0f}.")
notes.append(f"Context length (words) median={df['context_word_len'].median():.0f}, p95={df['context_word_len'].quantile(0.95):.0f}.")
notes.append(f"Response length (words) median={df['response_word_len'].median():.0f}, p95={df['response_word_len'].quantile(0.95):.0f}.")

hard_rate = (df["difficulty_score"] >= 3).mean()
notes.append(f"Hard-case heuristic (score>=3): {hard_rate:.1%} of the dataset.")

notes_path = os.path.join(OUTPUT_DIR, "day1_notes.json")
with open(notes_path, "w", encoding="utf-8") as f:
    json.dump({"notes": notes}, f, ensure_ascii=False, indent=2)

for i, n in enumerate(notes, 1):
    print(f"{i}. {n}")

print("Notes saved to:", notes_path)

1. Dataset size: 15011 rows, 8 categories.
2. Context availability: 29.8% of samples contain non-empty context.
3. Category imbalance: top category ratio = 24.9% (category: open_qa).
4. Instruction length (words) median=10, p95=27.
5. Context length (words) median=0, p95=277.
6. Response length (words) median=31, p95=202.
7. Hard-case heuristic (score>=3): 0.4% of the dataset.
Notes saved to: data_exploration/day1_notes.json


#chat gpt with zero shot

In [ ]:
# ---------- Build zero-shot prompt ----------

def build_zero_shot_prompt(instruction: str, context: str) -> str:
    """
    Create a clean zero-shot prompt.
    If context exists, include it.
    Otherwise, only use instruction.
    """

    instruction = (instruction or "").strip()
    context = (context or "").strip()

    if context:
        prompt = (
            "You are a helpful assistant.\n\n"
            "Instruction:\n"
            f"{instruction}\n\n"
            "Context:\n"
            f"{context}\n\n"
            "Answer:"
        )
    else:
        prompt = (
            "You are a helpful assistant.\n\n"
            "Instruction:\n"
            f"{instruction}\n\n"
            "Answer:"
        )

    return prompt

In [ ]:
import os
import time
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = "open api key"

# Initialize the OpenAI client
client = OpenAI()

def get_chatgpt_response(prompt: str, model_name: str = "gpt-4o-mini"):
    """
    Send a prompt to ChatGPT and return the generated answer.
    """

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.3  # low randomness for stable instruction-following
    )

    return response.choices[0].message.content.strip()

In [ ]:
test_prompt = "Explain what artificial intelligence is in one short paragraph."
print(get_chatgpt_response(test_prompt))

Artificial intelligence (AI) refers to the simulation of human intelligence processes by computer systems, enabling machines to perform tasks that typically require human cognitive functions. These processes include learning from data (machine learning), reasoning and problem-solving, understanding natural language, and perception. AI can be categorized into narrow AI, which is designed for specific tasks, and general AI, which aims to perform any intellectual task that a human can do. The goal of AI is to create systems that can operate autonomously and improve their performance over time through experience.


In [ ]:
# ---------- Run zero-shot on prompt sample ----------

zero_shot_outputs = []

for idx, row in prompt_df.iterrows():

    prompt = build_zero_shot_prompt(
        instruction=row["instruction"],
        context=row["context"]
    )

    try:
        answer = get_chatgpt_response(prompt)
    except Exception as e:
        print(f"Error at index {idx}: {e}")
        answer = "ERROR"

    zero_shot_outputs.append({
        "instruction": row["instruction"],
        "context": row["context"],
        "category": row["category"],
        "reference_response": row["response"],
        "model_response": answer
    })

    time.sleep(0.8)

    if idx % 20 == 0:
        print(f"Processed {idx} samples...")

zero_shot_df = pd.DataFrame(zero_shot_outputs)

print("Zero-shot completed.")
print("Total rows:", len(zero_shot_df))

Processed 0 samples...
Processed 20 samples...
Processed 40 samples...
Processed 60 samples...
Processed 80 samples...
Processed 100 samples...
Processed 120 samples...
Processed 140 samples...
Processed 160 samples...
Processed 180 samples...
Zero-shot completed.
Total rows: 200


In [ ]:
zero_shot_df.to_csv("zero_shot_results.csv", index=False)
print("File saved successfully.")

File saved successfully.


#chat gpt with few shot

In [ ]:

few_shot_examples = df.sample(2, random_state=42)

example1 = few_shot_examples.iloc[0]
example2 = few_shot_examples.iloc[1]

example1, example2

(instruction              What are the best selling albums of all time? ...
 context                                                                   
 response                 Best selling albums of all time:\n\n- Michael ...
 category                                                     brainstorming
 context_clean                                                             
 has_context                                                          False
 instruction_char_len                                                   116
 response_char_len                                                      378
 context_char_len                                                         0
 instruction_word_len                                                    25
 response_word_len                                                       68
 context_word_len                                                         0
 flag_long_context                                                    False
 flag_long_i

In [ ]:
def build_few_shot_prompt(instruction: str, context: str) -> str:
    """
    Create a few-shot prompt with two demonstration examples.
    """

    instruction = (instruction or "").strip()
    context = (context or "").strip()

    prompt = f"""
You are a helpful assistant that follows instructions carefully.

Example 1
Instruction:
{example1["instruction"]}

Answer:
{example1["response"]}


Example 2
Instruction:
{example2["instruction"]}

Answer:
{example2["response"]}


Now answer the following instruction.

Instruction:
{instruction}
"""

    if context:
        prompt += f"""

Context:
{context}
"""

    prompt += "\nAnswer:"

    return prompt

In [ ]:
few_shot_outputs = []

for idx, row in prompt_df.iterrows():

    prompt = build_few_shot_prompt(
        instruction=row["instruction"],
        context=row["context"]
    )

    try:
        answer = get_chatgpt_response(prompt)
    except Exception as e:
        print(f"Error at index {idx}: {e}")
        answer = "ERROR"

    few_shot_outputs.append({
        "instruction": row["instruction"],
        "context": row["context"],
        "category": row["category"],
        "reference_response": row["response"],
        "model_response": answer
    })

    time.sleep(0.8)

    if idx % 20 == 0:
        print(f"Processed {idx} samples...")


few_shot_df = pd.DataFrame(few_shot_outputs)

print("Few-shot completed.")
print("Total rows:", len(few_shot_df))

Processed 0 samples...
Processed 20 samples...
Processed 40 samples...
Processed 60 samples...
Processed 80 samples...
Processed 100 samples...
Processed 120 samples...
Processed 140 samples...
Processed 160 samples...
Processed 180 samples...
Few-shot completed.
Total rows: 200


In [ ]:
few_shot_df.to_csv("few_shot_results.csv", index=False)
print("Few-shot results saved.")

Few-shot results saved.


#eval using bleu+llm judge

In [ ]:
import pandas as pd

zero_df = pd.read_csv("zero_shot_results.csv")
few_df = pd.read_csv("few_shot_results.csv")

print(len(zero_df), len(few_df))

200 200


In [ ]:
eval_df.head()

,instruction,context,response,category,context_clean,has_context,instruction_char_len,response_char_len,context_char_len,instruction_word_len,response_word_len,context_word_len,flag_long_context,flag_long_instruction,flag_long_response,flag_no_context,difficulty_score
0,When running a marathon and attempting to run ...,,Fast-digesting carbohydrates are useful to mak...,classification,,False,243,460,0,36,78,0,False,True,False,True,2
1,Classify each of the following as a ridesharin...,,Doordash: Food Delivery\nLyft: Ridesharing\nWo...,classification,,False,124,143,0,19,18,0,False,False,False,True,1
2,Classify these as positive or negative reviews...,,"""Interesting concept but execution left a lot ...",classification,,False,245,237,0,42,39,0,False,True,False,True,2
3,Give me the list of British artists born befor...,,These are the British artists born before 1700...,brainstorming,,False,52,1889,0,10,287,0,False,False,True,True,2
4,"Give me a title of one of the ""Harry Potter"" n...",,Harry Potter and the Half-Blood Prince,open_qa,,False,52,38,0,11,6,0,False,False,False,True,1


In [ ]:
eval_df = eval_df.merge(
    zero_df[["instruction","model_response"]],
    on="instruction",
    how="left"
).rename(columns={"model_response":"zero_response"})

eval_df = eval_df.merge(
    few_df[["instruction","model_response"]],
    on="instruction",
    how="left"
).rename(columns={"model_response":"few_response"})

In [ ]:
!pip install sacrebleu
import sacrebleu

references = [[r] for r in eval_df["response"]]

zero_bleu = sacrebleu.corpus_bleu(
    eval_df["zero_response"].tolist(),
    references
)

few_bleu = sacrebleu.corpus_bleu(
    eval_df["few_response"].tolist(),
    references
)

print("Zero-shot BLEU:", zero_bleu.score)
print("Few-shot BLEU:", few_bleu.score)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.9 MB/s eta 0:00:00
Zero-shot BLEU: 21.41301553535276
Few-shot BLEU: 12.090340630072072


In [ ]:
from openai import OpenAI
client = OpenAI()

def judge_answer(instruction, reference, answer):

    prompt = f"""
You are evaluating instruction-following quality.

Instruction:
{instruction}

Reference Answer:
{reference}

Model Answer:
{answer}

Score the answer from 1 to 10 based on how well it follows the instruction.
Return only the number.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"user","content":prompt}],
        temperature=0
    )

    return response.choices[0].message.content.strip()

In [ ]:
zero_scores = []
few_scores = []

for i,row in eval_df.iterrows():

    z = judge_answer(
        row["instruction"],
        row["response"],
        row["zero_response"]
    )

    f = judge_answer(
        row["instruction"],
        row["response"],
        row["few_response"]
    )

    zero_scores.append(float(z))
    few_scores.append(float(f))

    if i % 5 == 0:
        print("processed", i)

processed 0
processed 5
processed 10
processed 15
processed 20
processed 25
processed 30
processed 35


In [ ]:
eval_df["zero_score"] = zero_scores
eval_df["few_score"] = few_scores

print("Average Zero-shot:", eval_df["zero_score"].mean())
print("Average Few-shot:", eval_df["few_score"].mean())

Average Zero-shot: 9.1
Average Few-shot: 8.6


In [ ]:
eval_df.to_csv("evaluation_results.csv", index=False)

#fine tune model

In [ ]:
!pip install transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 25.9 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset


# take small subset for fine tuning
train_df = df.sample(400, random_state=42)

In [ ]:
def format_example(row):

    instruction = row["instruction"]
    context = row["context"]
    response = row["response"]

    if context:
        prompt = f"User: {instruction}\nContext: {context}\nAssistant:"
    else:
        prompt = f"User: {instruction}\nAssistant:"

    full_text = prompt + " " + response

    return full_text


train_texts = train_df.apply(format_example, axis=1).tolist()

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "text": train_texts
})

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="dolly_finetuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2169 > 2048). Running this sequence through the model will result in indexing errors


Truncating train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,2.025786
20,1.974442
30,1.889976
40,1.963467
50,1.915243


TrainOutput(global_step=50, training_loss=1.9537826156616211, metrics={'train_runtime': 263.9672, 'train_samples_per_second': 1.515, 'train_steps_per_second': 0.189, 'total_flos': 771944436817920.0, 'train_loss': 1.9537826156616211})

In [ ]:
trainer.model.save_pretrained("dolly_lora_model")
tokenizer.save_pretrained("dolly_lora_model")

('dolly_lora_model/tokenizer_config.json',
 'dolly_lora_model/chat_template.jinja',
 'dolly_lora_model/tokenizer.json')

#eval fine tune and compare

In [ ]:
import os
import re
import json
import time
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

zero_df = pd.read_csv("zero_shot_results.csv")   # 200 rows
few_df  = pd.read_csv("few_shot_results.csv")    # 200 rows

print("zero_df:", len(zero_df), "few_df:", len(few_df))



print("eval_df rows:", len(eval_df))
eval_df.head(2)

zero_df: 200 few_df: 200
eval_df rows: 40


,instruction,context,response,category,context_clean,has_context,instruction_char_len,response_char_len,context_char_len,instruction_word_len,...,context_word_len,flag_long_context,flag_long_instruction,flag_long_response,flag_no_context,difficulty_score,zero_response,few_response,zero_score,few_score
0,When running a marathon and attempting to run ...,,Fast-digesting carbohydrates are useful to mak...,classification,,False,243,460,0,36,...,0,False,True,False,True,2,1. Fast digesting carbohydrates - Useful\n2. A...,- Fast digesting carbohydrates: Useful\n- A fi...,4.0,3.0
1,Classify each of the following as a ridesharin...,,Doordash: Food Delivery\nLyft: Ridesharing\nWo...,classification,,False,124,143,0,19,...,0,False,False,False,True,1,Here is the classification of each company:\n\...,Ridesharing companies:\n- Lyft\n- Bolt\n- Didi...,10.0,8.0


In [ ]:
!pip -q install transformers peft accelerate bitsandbytes

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
LORA_DIR = "dolly_lora_model"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

model = PeftModel.from_pretrained(base_model, LORA_DIR)
model.eval()

def build_model_prompt(instruction: str, context: str) -> str:
    """
    Prompt format for the fine-tuned model (keep consistent with Day 5).
    """
    instruction = (instruction or "").strip()
    context = (context or "").strip()

    if context:
        return f"User: {instruction}\nContext: {context}\nAssistant:"
    else:
        return f"User: {instruction}\nAssistant:"

@torch.no_grad()
def generate_batch(prompts, max_new_tokens=160):
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0,
        top_p=1.0,
        pad_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

    answers = []
    for full_text, prompt in zip(decoded, prompts):
        ans = full_text[len(prompt):].strip() if full_text.startswith(prompt) else full_text.strip()

        ans = ans.split("User:")[0].strip()
        answers.append(ans)
    return answers

BATCH_SIZE = 4
finetuned_outputs = []

prompts = []
for _, r in eval_df.iterrows():
    prompts.append(build_model_prompt(r["instruction"], r["context"]))

for start in range(0, len(prompts), BATCH_SIZE):
    batch_prompts = prompts[start:start+BATCH_SIZE]
    batch_answers = generate_batch(batch_prompts, max_new_tokens=160)

    for i, ans in enumerate(batch_answers):
        row = eval_df.iloc[start+i]
        finetuned_outputs.append({
            "instruction": (row["instruction"] or "").strip(),
            "context": (row["context"] or "").strip(),
            "category": row["category"],
            "reference_response": row["response"],
            "finetuned_response": ans
        })

    print(f"Generated {min(start+BATCH_SIZE, len(prompts))}/{len(prompts)}")

finetuned_df = pd.DataFrame(finetuned_outputs)
finetuned_df.to_csv("finetuned_results_eval40.csv", index=False)
print("Saved:", "finetuned_results_eval40.csv")
finetuned_df.head(2)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Generated 4/40


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


...

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Generated 32/40


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Generated 36/40
Generated 40/40
Saved: finetuned_results_eval40.csv


,instruction,context,category,reference_response,finetuned_response
0,When running a marathon and attempting to run ...,,classification,Fast-digesting carbohydrates are useful to mak...,Useful\nDangerous\nNeither\nFast digesting car...
1,Classify each of the following as a ridesharin...,,classification,Doordash: Food Delivery\nLyft: Ridesharing\nWo...,1. Doordash\n2. Lyft\n3. Wolt\n4. Deliveroo\n5...


In [ ]:
!pip -q install sacrebleu
import sacrebleu

def corpus_bleu(references, hypotheses):
    refs = [[r] for r in references]
    return sacrebleu.corpus_bleu(hypotheses, refs).score

ref = finetuned_df["reference_response"].fillna("").tolist()
ft_hyp = finetuned_df["finetuned_response"].fillna("").tolist()

finetuned_bleu = corpus_bleu(ref, ft_hyp)
print(f"Fine-tuned BLEU (Eval-40): {finetuned_bleu:.2f}")

Fine-tuned BLEU (Eval-40): 5.12


In [ ]:
from openai import OpenAI
client = OpenAI()

JUDGE_MODEL = "gpt-4o-mini"

def safe_str(x):
    return ("" if pd.isna(x) else str(x)).strip()

def build_judge_prompt(instruction, context, reference, model_answer):
    instruction = safe_str(instruction)
    context = safe_str(context)
    reference = safe_str(reference)
    model_answer = safe_str(model_answer)

    context_block = f"Context:\n{context}\n\n" if context else ""

    return (
        "You are an impartial evaluator for instruction-following.\n"
        "Score how well the model answer follows the instruction.\n"
        "Return ONLY valid JSON with keys: score, follows_instruction, reason.\n"
        "score must be an integer 1-10.\n"
        "follows_instruction must be true/false.\n"
        "reason must be short.\n\n"
        f"Instruction:\n{instruction}\n\n"
        f"{context_block}"
        f"Reference Answer:\n{reference}\n\n"
        f"Model Answer:\n{model_answer}\n"
    )

def judge_once(instruction, context, reference, model_answer, retries=3):
    prompt = build_judge_prompt(instruction, context, reference, model_answer)

    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role":"user","content":prompt}],
                temperature=0.0
            )
            txt = resp.choices[0].message.content.strip()
            data = json.loads(txt)
            score = int(data["score"])
            follows = bool(data["follows_instruction"])
            reason = str(data["reason"]).strip()
            score = max(1, min(10, score))
            return score, follows, reason
        except Exception as e:
            if attempt == retries - 1:
                return None, None, f"JUDGE_ERROR: {e}"
            time.sleep(1.0)

zero_map = {safe_str(r["instruction"]): safe_str(r["model_response"]) for _, r in zero_df.iterrows()}
few_map  = {safe_str(r["instruction"]): safe_str(r["model_response"]) for _, r in few_df.iterrows()}

eval_compare_rows = []
for _, r in finetuned_df.iterrows():
    inst = safe_str(r["instruction"])
    eval_compare_rows.append({
        "instruction": inst,
        "context": safe_str(r["context"]),
        "category": r["category"],
        "reference_response": safe_str(r["reference_response"]),
        "zero_response": zero_map.get(inst, ""),
        "few_response": few_map.get(inst, ""),
        "finetuned_response": safe_str(r["finetuned_response"]),
    })

eval_compare_df = pd.DataFrame(eval_compare_rows)

# ---- Run judge for all three: zero, few, finetuned ----
judge_rows = []
for i, r in eval_compare_df.iterrows():
    z_score, z_ok, z_reason = judge_once(r["instruction"], r["context"], r["reference_response"], r["zero_response"])
    f_score, f_ok, f_reason = judge_once(r["instruction"], r["context"], r["reference_response"], r["few_response"])
    t_score, t_ok, t_reason = judge_once(r["instruction"], r["context"], r["reference_response"], r["finetuned_response"])

    judge_rows.append({
        "category": r["category"],
        "instruction": r["instruction"],
        "zero_score": z_score, "zero_ok": z_ok, "zero_reason": z_reason,
        "few_score":  f_score, "few_ok":  f_ok, "few_reason":  f_reason,
        "finetuned_score": t_score, "finetuned_ok": t_ok, "finetuned_reason": t_reason,
    })

    if (i + 1) % 5 == 0:
        print(f"Judged {i+1}/{len(eval_compare_df)}")

judge6_df = pd.DataFrame(judge_rows)
judge6_df.to_csv("day6_judge_comparison_eval40.csv", index=False)
print("Saved:", "day6_judge_comparison_eval40.csv")

print("Avg Zero-shot:", judge6_df["zero_score"].dropna().mean())
print("Avg Few-shot :", judge6_df["few_score"].dropna().mean())
print("Avg Fine-tuned:", judge6_df["finetuned_score"].dropna().mean())

Judged 5/40
Judged 10/40
Judged 15/40
Judged 20/40
Judged 25/40
Judged 30/40
Judged 35/40
Judged 40/40
Saved: day6_judge_comparison_eval40.csv
Avg Zero-shot: 9.61111111111111
Avg Few-shot : 9.333333333333334
Avg Fine-tuned: 3.75


In [ ]:
summary = {
    "bleu_finetuned_eval40": float(finetuned_bleu),
    "judge_avg_zero": float(judge6_df["zero_score"].dropna().mean()),
    "judge_avg_few": float(judge6_df["few_score"].dropna().mean()),
    "judge_avg_finetuned": float(judge6_df["finetuned_score"].dropna().mean()),
    "judge_follow_rate_zero": float(judge6_df["zero_ok"].dropna().mean()),
    "judge_follow_rate_few": float(judge6_df["few_ok"].dropna().mean()),
    "judge_follow_rate_finetuned": float(judge6_df["finetuned_ok"].dropna().mean()),
}

comparison_table = pd.DataFrame([
    {"method": "zero-shot (GPT)", "judge_avg": summary["judge_avg_zero"], "follow_rate": summary["judge_follow_rate_zero"]},
    {"method": "few-shot (GPT)",  "judge_avg": summary["judge_avg_few"],  "follow_rate": summary["judge_follow_rate_few"]},
    {"method": "fine-tuned (LoRA)", "judge_avg": summary["judge_avg_finetuned"], "follow_rate": summary["judge_follow_rate_finetuned"]},
])

comparison_table.to_csv("day6_final_comparison.csv", index=False)

with open("day6_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved: day6_final_comparison.csv")
print("Saved: day6_summary.json")
comparison_table

Saved: day6_final_comparison.csv
Saved: day6_summary.json


,method,judge_avg,follow_rate
0,zero-shot (GPT),9.611111,0.972222
1,few-shot (GPT),9.333333,0.948718
2,fine-tuned (LoRA),3.750000,0.166667
